In [1]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchmetrics.classification import AUROC
from torchvision.datasets import Imagenette, SVHN
from torchvision import models

import logging

import importlib
import FL_sim

importlib.reload(FL_sim)
from FL_sim import FLSimulator, FederatedModelWrapper

torch.set_float32_matmul_precision('high')

In [2]:
class ResNetPLModel(FederatedModelWrapper):
    def __init__(self, num_classes, resnet_version='resnet50', lr=0.01,
                 logging_disabled=False, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.lr = lr
        self.resnet_version = resnet_version
        self.num_classes = num_classes
        self.logging_disabled = logging_disabled

        # 1) backbone ----------------------------------------------------------
        backbone = dict(resnet50=models.resnet50,
                        resnet18=models.resnet18)[resnet_version](weights=None)

        # 2) replace the classification head ----------------------------------
        in_features = backbone.fc.in_features
        backbone.fc = nn.Linear(in_features, num_classes)

        self.model = backbone

        # 3) loss & metric -----------------------------------------------------
        self.loss_fn = nn.CrossEntropyLoss()
        self.aucroc = AUROC(num_classes=num_classes, average="weighted", task="multiclass")

    def forward(self, x):
        return self.model(x)

    def step_with_custom_logs(self, stage: str, batch, batch_idx: int):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)

        auc = None
        if stage is None or not self.logging_disabled:
            auc = self.aucroc(torch.softmax(logits.detach(), dim=1), y)

        if not self.logging_disabled:
            self.log(f"{stage}_loss", loss, on_step=True, prog_bar=True, logger=True)
            self.log(f"{stage}_auc", auc,
                     on_step=True, prog_bar=True, logger=True)

        return loss, auc

    def training_step(self, batch, batch_idx):
        self.batch_idx = batch_idx
        loss, auc = self.step_with_custom_logs('train', batch, batch_idx)
        return loss

    def validation_step(self, batch, batch_idx):
        loss, auc = self.step_with_custom_logs('valid', batch, batch_idx)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.SGD(self.model.parameters(), lr=self.lr, momentum=0.9)
        return optimizer

    def clone(self, copy=None):
        new_model = ResNetPLModel(num_classes=self.num_classes, lr=self.lr,
                                  resnet_version=self.resnet_version,
                                  logging_disabled=self.logging_disabled)
        new_model.load_state_dict(self.state_dict())

        return super(ResNetPLModel, self).clone(copy=new_model)

In [3]:
# dataset = [
#     Imagenette(
#         root='./data/Imagenet', split=s,
#         transform=transforms.Compose([
#             transforms.Resize(256),
#             transforms.CenterCrop(224),
#             transforms.ToTensor(),
#             transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
#         ])
#     ) for s in ['train', 'val']]

dataset = [
    SVHN(
        root='./data/SVHN', split=s,
        transform=transforms.Compose([
            transforms.Resize(32),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    ) for s in ['train', 'test']]

In [4]:
# dataset = [torch.utils.data.Subset(d, list(range(50))) for d in dataset]

In [5]:
def pre_send_process(single_model_grad_agent):
    return single_model_grad_agent


def server_rec_process(all_encoded_model_grad):
    return all_encoded_model_grad

In [6]:
k = 2  # Number of workers
bs = 14000
#bs /= 50 # imagenet
# bs /= 3 # resnet 50
bs = int(bs)

sim = FLSimulator(
    num_agents=k, communication_rounds=3, client_epochs_per_round=20,
    batch_size=bs, dataset_train=dataset[0], dataset_test=dataset[1],
    pl_model=ResNetPLModel(num_classes=10, resnet_version='resnet18',
                           record_gradients=True, logging_disabled=True),
    aggregation_method='fedavg', iid_data=True,
    pre_send_process=pre_send_process,
    server_rec_process=server_rec_process,
)

In [7]:
logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)

sim.run_simulation()

round 1/3


GPU available: True (cuda), used: True


         loss: 2.5606861114501953
         auc: 0.4988822340965271
  > training agent 1/2


TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
D:\User\Software\MiniConda3\envs\SuperCondaEnv\Lib\site-packages\pytorch_lightning\trainer\configuration_validator.py:74: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
`Trainer.fit` stopped: `max_epochs=20` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


  > training agent 2/2


...

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


         loss: 0.6592416763305664
         auc: 0.9732509851455688
  > training agent 1/2


D:\User\Software\MiniConda3\envs\SuperCondaEnv\Lib\site-packages\pytorch_lightning\trainer\call.py:54: Detected KeyboardInterrupt, attempting graceful shutdown...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


  > training agent 2/2


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!